In [14]:
from langgraph.graph import StateGraph
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from typing import TypedDict
from langgraph.constants import END, START
from pydantic import BaseModel, Field
from typing import Annotated
import operator

In [15]:
load_dotenv()

True

In [16]:
essay = """ Artificial Intelligence, commonly known as AI, has emerged as one of the most transformative technologies of the 21st century. From smartphones and search engines to healthcare, education, transportation, and business, AI is becoming a part of everyday life. What once seemed like science fiction is now being used to solve real-world problems, automate tasks, and assist people in making better decisions. The rapid rise of AI is changing not only how we work, but also how we learn, communicate, and interact with technology.

The idea of AI is not new. Researchers began exploring the possibility of machines that could imitate human intelligence during the 20th century. However, AI remained limited for many years because computers lacked sufficient computing power, data, and advanced algorithms. The situation changed dramatically with the growth of the internet, cloud computing, powerful processors, and large datasets. Advances in machine learning and deep learning allowed computers to recognize patterns, understand language, analyze images, and make predictions with remarkable accuracy.

One of the biggest reasons behind the recent rise of AI is generative AI. Systems capable of generating text, images, audio, video, and computer code have made AI accessible to ordinary users. AI assistants can help students understand difficult concepts, developers write and debug code, businesses analyze information, and creators produce content. Instead of simply following fixed instructions, modern AI systems can respond intelligently to natural-language requests and adapt their output to different situations.

AI is also transforming industries. In healthcare, AI can assist doctors by analyzing medical images, identifying patterns, and supporting research into new treatments. In agriculture, it can help monitor crops, predict weather-related risks, and improve the use of water and fertilizers. In transportation, AI supports navigation, traffic prediction, driver-assistance systems, and the development of autonomous vehicles. In finance and business, it is used for fraud detection, forecasting, customer support, and data analysis. These applications demonstrate that AI is not limited to one field; it has the potential to influence almost every sector of society.

Education is another area where AI is creating significant opportunities. Students can use AI-powered tools as personalized learning assistants that explain concepts, generate practice questions, and provide feedback. Teachers can use AI to prepare educational material and analyze learning patterns. However, students must use these tools responsibly. Relying completely on AI for assignments can reduce critical thinking and creativity. Therefore, AI should support education rather than replace genuine learning.

Despite its advantages, the rise of AI also creates serious challenges. Automation may change the nature of many jobs, particularly tasks that are repetitive and predictable. This does not necessarily mean that humans will become unnecessary, but it does mean that workers may need to develop new skills. Privacy is another concern because AI systems often depend on large amounts of data. There are also risks related to misinformation, deepfakes, algorithmic bias, and the misuse of AI.

For this reason, responsible development of AI is extremely important. Governments, technology companies, educational institutions, and society must work together to establish appropriate rules and ethical standards. AI systems should be transparent, secure, fair, and designed with human well-being in mind. Human judgment must remain important, especially in sensitive areas such as healthcare, law, employment, and public safety.

The future of AI will depend largely on how humans choose to use it. AI should not be viewed simply as a replacement for human intelligence. Instead, it can be seen as a powerful tool that can extend human abilities. Creativity, empathy, judgment, leadership, and ethical reasoning remain uniquely important human qualities. The strongest future may come from collaboration between humans and intelligent machines rather than competition between them.

In conclusion, the rise of Artificial Intelligence is one of the defining technological developments of our time. It offers enormous opportunities for innovation, productivity, education, and social development, while also presenting challenges that cannot be ignored. The goal should not be to stop the progress of AI, but to guide it responsibly. If developed and used wisely, AI can become a powerful partner in building a more efficient, innovative, and sustainable future."""

In [17]:
model = ChatOllama(
    model="qwen3:1.7b",
    temperature=0
)

In [18]:
class Rules(BaseModel):

    feedback: str  = Field(description="Detailed feedback for an essay",)
    score: int = Field(description="Score out of 10")
    

In [19]:
structured_output = model.with_structured_output(Rules, method = "json_schema")


In [20]:
class UPSC(TypedDict):

    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    individual_scores: Annotated[list[int], operator.add]
    final_feedback: str
    final_score: float

In [21]:
def language_feedback(state: UPSC):

    prompt = f"Evaluate the language quality of the {state['essay']} and provide a feedback and score out of 10"
    output = structured_output.invoke(prompt)

    return {'language_feedback': output.feedback, 'individual_scores': [output.score]}

In [22]:
def analysis_feedback(state: UPSC):

    prompt = f"Evaluate the depth of analysis quality of the {state['essay']} and provide a feedback and score out of 10"
    output = structured_output.invoke(prompt)

    return {'analysis_feedback': output.feedback, 'individual_scores': [output.score]}

In [23]:
def clarity_feedback(state: UPSC):

    prompt = f"Evaluate the clarity of thought quality of the {state['essay']} and provide a feedback and score out of 10"
    output = structured_output.invoke(prompt)

    return {'clarity_feedback': output.feedback, 'individual_scores': [output.score]}

In [24]:
def final_feedback(state: UPSC):

    prompt = f"Evaluate the following feedbacks \n language_feedback = {state['language_feedback']} \n analysis_feedback = {state['analysis_feedback']} \n clarity_feedback = {state['clarity_feedback']} and provide a final feedback"
    final_output = structured_output.invoke(prompt)
    final_score = sum(state['individual_scores'])/len(state['individual_scores'])

    return {'final_feedback': final_output.feedback, 'final_score': final_score}

In [25]:
graph = StateGraph(UPSC)

graph.add_node('language_feedback', language_feedback)
graph.add_node('analysis_feedback', analysis_feedback)
graph.add_node('clarity_feedback', clarity_feedback)
graph.add_node('final_feedback', final_feedback)

graph.add_edge(START, 'language_feedback')
graph.add_edge(START, 'analysis_feedback')
graph.add_edge(START, 'clarity_feedback')

graph.add_edge('language_feedback', 'final_feedback')
graph.add_edge('analysis_feedback', 'final_feedback')
graph.add_edge('clarity_feedback', 'final_feedback')

graph.add_edge('final_feedback', END)

workflow = graph.compile()


In [26]:
workflow.invoke({'essay': essay})

{'essay': ' Artificial Intelligence, commonly known as AI, has emerged as one of the most transformative technologies of the 21st century. From smartphones and search engines to healthcare, education, transportation, and business, AI is becoming a part of everyday life. What once seemed like science fiction is now being used to solve real-world problems, automate tasks, and assist people in making better decisions. The rapid rise of AI is changing not only how we work, but also how we learn, communicate, and interact with technology.\n\nThe idea of AI is not new. Researchers began exploring the possibility of machines that could imitate human intelligence during the 20th century. However, AI remained limited for many years because computers lacked sufficient computing power, data, and advanced algorithms. The situation changed dramatically with the growth of the internet, cloud computing, powerful processors, and large datasets. Advances in machine learning and deep learning allowed co